# Comparação com o dataset ouro

Compara o ouro (`avaliacao/ouro_final/ouro_lc95.json`, 376 leis) com os quatro métodos: o **regex** (`coleta/`) e os três LLMs da rodada j24k (**Qwen**, **Gemma** e **Llama**, pastas `_junta` em `modelos/resultados/lc95/`).

**Por dispositivo** (artigo, parágrafo, inciso, alínea, item, agrupadores), duas métricas:

- **similaridade**: Levenshtein normalizada, `1 - distância / maior comprimento` (1 = texto idêntico);
- **acerto**: similaridade ≥ 0,95. A **acurácia** é a fração de acertos.

**Casamento.** O dispositivo do método é casado com o do ouro pelo caminho sem os agrupadores (ex.: `Art. 2º, § 1º, a)`). Quando falta, conta como 0 e erro. O dispositivo que o método inventou (a "sobra") também conta como 0 no denominador da lei. A lei sem JSON válido vale 0. Esse é o casamento **estrito**, que cobra o texto e a posição na árvore.

O casamento **flexível** (colunas `_flex`) dá aos dispositivos sem par a sobra de texto mais parecido (similaridade ≥ 0,5). Ele mede só o texto. A diferença entre os dois é o custo dos erros de hierarquia e de tipo.

**Global.** Primeiro a média por lei. Depois a estimativa estratificada para a população (15.754 leis), com IC de 95%. Os métodos são comparados com o teste de Friedman e com o Wilcoxon pareado com correção de Holm.

Roda a partir da raiz do repositório ou de `metricas/`. Dependências: `pandas`, `rapidfuzz`, `scipy` e `xlsxwriter`.

In [1]:
import os, pathlib, sys
if pathlib.Path.cwd().name == "metricas":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from metricas import metricas_ouro as mo

pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## 1. Leitura das fontes

In [2]:
amostra, ouro, metodos = mo.carregar()
print("leis na amostra:", len(amostra), "| no ouro:", len(ouro))
for m, leis in metodos.items():
    print(f"{m:6s} leis com JSON válido: {sum(v is not None for v in leis.values())}")

leis na amostra: 376 | no ouro: 376
regex  leis com JSON válido: 376
qwen   leis com JSON válido: 376
gemma  leis com JSON válido: 368
llama  leis com JSON válido: 373


## 2. Comparação dispositivo a dispositivo

In [3]:
tabelas = mo.comparar(amostra, ouro, metodos)
disp = tabelas["dispositivos"]
print("dispositivos do ouro:", len(disp), "| sobras (todos os métodos):", len(tabelas["sobras"]))
disp[["slug", "caminho", "texto_ouro", "sim_regex", "sim_qwen", "sim_gemma", "sim_llama"]].head(10)

dispositivos do ouro: 3508 | sobras (todos os métodos): 6041


,slug,caminho,texto_ouro,sim_regex,sim_qwen,sim_gemma,sim_llama
0,lei-164-2-janeiro-1936-555951,Art. 1º,Fica desincorporado e entregue ao Estado de Mi...,1.0,0.9950,1.0,1.0
1,lei-164-2-janeiro-1936-555951,Art. 2º,A entrega dos bens a que se refere o artigo an...,1.0,0.9931,1.0,0.0
2,lei-164-2-janeiro-1936-555951,Art. 3º,O Governo da União continuará a contribuir com...,1.0,1.0000,1.0,0.0
3,lei-164-2-janeiro-1936-555951,Art. 4º,"Ao Estado de Minas Geraes, competirão, desde q...",1.0,0.9943,1.0,0.0
4,lei-164-2-janeiro-1936-555951,Art. 5º,"Os funccionários effectivos, contractados e de...",1.0,0.9884,1.0,0.0
5,lei-164-2-janeiro-1936-555951,Art. 6º,Aquelles que tiverem de ser dispensados por co...,1.0,1.0000,1.0,0.0
6,lei-164-2-janeiro-1936-555951,Art. 7º,O presente decreto entrará em vigor desde a da...,1.0,1.0000,1.0,0.0
7,lei-164-2-janeiro-1936-555951,Art. 8º,Revogam-se as disposições em contrario.,1.0,1.0000,1.0,0.0
8,lei-3139-2-agosto-1916-574077,CAPITULO I,DOS ELEITORES,1.0,1.0000,1.0,1.0
9,lei-3139-2-agosto-1916-574077,"CAPITULO I, Art. 1º",Só terão voto nas eleições federaes e nas loca...,1.0,0.9929,1.0,1.0


## 3. Resultado global

`similaridade` e `acuracia` são as estimativas estratificadas do casamento estrito, com IC de 95%. As colunas `_flex` vêm do casamento flexível. As colunas `_por_dispositivo` somam todos os dispositivos, cada um com o peso da sua lei, em vez de fazer a média por lei.

In [4]:
res = mo.resumo(tabelas, amostra)
res.set_index("metodo")[["posicao", "leis_json_valido",
    "similaridade", "sim_ic95_inf", "sim_ic95_sup", "acuracia", "acc_ic95_inf", "acc_ic95_sup",
    "similaridade_flex", "acuracia_flex", "similaridade_por_dispositivo", "acuracia_por_dispositivo",
    "faltando", "sobrando", "faltando_flex", "sobrando_flex"]].T

metodo,regex,gemma,qwen,llama
posicao,1.0000,2.0000,3.0000,4.0000
leis_json_valido,376.0000,368.0000,376.0000,373.0000
similaridade,0.9670,0.8010,0.7424,0.5216
sim_ic95_inf,0.9522,0.7704,0.7156,0.4843
sim_ic95_sup,0.9818,0.8316,0.7692,0.5589
acuracia,0.9641,0.7917,0.7271,0.5112
acc_ic95_inf,0.9488,0.7602,0.6989,0.4735
acc_ic95_sup,0.9795,0.8232,0.7553,0.5489
similaridade_flex,0.9839,0.8466,0.8324,0.8038
acuracia_flex,0.9812,0.8356,0.8161,0.7881


## 4. Os métodos são diferentes? (Friedman e Wilcoxon pareado com Holm)

In [5]:
friedman, pares = mo.testes(tabelas, amostra)
friedman

,metrica,qui_quadrado,p_valor,leis
0,similaridade,394.5598,3.3382e-85,376
1,acuracia,376.0687,3.3764e-81,376
2,similaridade_flex,213.0112,6.5068e-46,376
3,acuracia_flex,180.8250,5.8521e-39,376


In [6]:
pares

,metrica,metodo_a,metodo_b,diferenca_a_menos_b,dif_ic95_inf,dif_ic95_sup,a_melhor_em,b_melhor_em,empates,wilcoxon_p,wilcoxon_p_holm,significativo_5pct
0,similaridade,regex,qwen,0.2246,0.1955,0.2536,191,12,173,4.0395e-28,1.6158e-27,sim
1,similaridade,regex,gemma,0.1660,0.1340,0.1980,156,11,209,7.4518e-22,2.2355e-21,sim
2,similaridade,regex,llama,0.4454,0.4058,0.4850,246,8,122,2.5725e-39,1.5435e-38,sim
3,similaridade,qwen,gemma,-0.0586,-0.0849,-0.0323,74,125,177,1.2369e-05,1.2369e-05,sim
4,similaridade,qwen,llama,0.2208,0.1827,0.2589,202,62,112,1.3320e-21,2.6641e-21,sim
5,similaridade,gemma,llama,0.2794,0.2423,0.3165,209,42,125,1.6141e-30,8.0704e-30,sim
6,acuracia,regex,qwen,0.2370,0.2067,0.2674,168,13,195,1.9517e-25,7.8067e-25,sim
7,acuracia,regex,gemma,0.1724,0.1396,0.2053,125,12,239,6.5264e-18,1.3053e-17,sim
8,acuracia,regex,llama,0.4529,0.4129,0.4929,234,7,135,1.7106e-37,1.0264e-36,sim
9,acuracia,qwen,gemma,-0.0646,-0.0909,-0.0383,59,114,203,5.0096e-06,5.0096e-06,sim


## 5. Por época, tamanho e tipo de dispositivo

In [7]:
mo.por_grupo(tabelas, amostra, "epoca")

similaridade                         acuracia                         similaridade_flex                         acuracia_flex                        
metodo           regex    qwen   gemma   llama    regex    qwen   gemma   llama             regex    qwen   gemma   llama         regex    qwen   gemma   llama
epoca                                                                                                                                                          
1891-1945       0.9088  0.7805  0.8758  0.6810   0.8788  0.7787  0.8752  0.6800            0.9088  0.9543  0.9417  0.8745        0.8788  0.9518  0.9410  0.8729
1946-1988       0.9432  0.7810  0.8065  0.6047   0.9408  0.7644  0.7993  0.5928            0.9780  0.8585  0.8408  0.8136        0.9759  0.8406  0.8332  0.7953
1989-2026       0.9970  0.6989  0.7883  0.4208   0.9962  0.6837  0.7760  0.4111            0.9970  0.7940  0.8438  0.7871        0.9962  0.7781  0.8282  0.7727

In [8]:
mo.por_grupo(tabelas, amostra, "tamanho")

similaridade                         acuracia                         similaridade_flex                         acuracia_flex                        
metodo         regex    qwen   gemma   llama    regex    qwen   gemma   llama             regex    qwen   gemma   llama         regex    qwen   gemma   llama
tamanho                                                                                                                                                      
grande        0.9398  0.5032  0.6914  0.3463   0.9377  0.4900  0.6822  0.3427            0.9682  0.7327  0.8182  0.7423        0.9662  0.7167  0.8038  0.7325
media         0.9671  0.8536  0.8638  0.5530   0.9640  0.8340  0.8538  0.5402            0.9894  0.8876  0.8739  0.8533        0.9866  0.8680  0.8638  0.8321
pequena       0.9940  0.8704  0.8478  0.6654   0.9907  0.8572  0.8391  0.6508            0.9940  0.8769  0.8478  0.8158        0.9907  0.8637  0.8391  0.7997

In [9]:
mo.por_tipo(tabelas).set_index("tipo").T

tipo,artigo,paragrafo,inciso,alinea,capitulo,secao,item
dispositivos,1928.0000,690.0000,598.0000,218.0000,65.0000,7.0000,2.0
sim_regex,0.9706,0.9583,0.9641,0.9901,0.9998,1.0000,0.0
acc_regex,0.9668,0.9580,0.9615,0.9862,1.0000,1.0000,0.0
sim_flex_regex,0.9911,0.9828,0.9758,0.9993,0.9998,1.0000,0.0
acc_flex_regex,0.9876,0.9826,0.9732,0.9954,1.0000,1.0000,0.0
faltando_regex,41.0000,28.0000,20.0000,2.0000,0.0000,0.0000,2.0
faltando_flex_regex,1.0000,11.0000,13.0000,0.0000,0.0000,0.0000,2.0
sim_qwen,0.8645,0.2841,0.4113,0.3771,0.8000,0.0000,1.0
acc_qwen,0.8356,0.2768,0.4013,0.3761,0.8000,0.0000,1.0
sim_flex_qwen,0.8829,0.7769,0.6435,0.6419,0.8615,0.5230,1.0


## 6. Campos da lei (epígrafe, ementa, preâmbulo, fecho)

Similaridade média e acurácia por campo, sem peso. Esses campos não entram na métrica global dos dispositivos.

In [10]:
mo.resumo_campos(tabelas)

,campo,regex,qwen,gemma,llama,acc_regex,acc_qwen,acc_gemma,acc_llama
0,epigrafe,1.0000,0.9905,0.9787,0.9920,1.0000,0.9761,0.9787,0.9920
1,ementa,0.9282,0.9940,0.9701,0.9918,0.9282,0.9894,0.9628,0.9920
2,preambulo,1.0000,0.9958,0.9762,0.9828,1.0000,0.9920,0.9734,0.9574
3,fecho,0.9839,0.9794,0.9579,0.9793,0.9840,0.9734,0.9574,0.9787


## 7. Gravação

A planilha `metricas/resultados/comparacao_ouro.xlsx` tem a aba `dispositivos`, com uma linha por dispositivo do ouro: o texto do ouro e, para cada método, o caminho, o texto, a forma do casamento, a similaridade e o acerto. As outras abas são o resumo, os testes, os recortes, `por_lei` e `sobras`.

In [11]:
arquivo = mo.exportar(tabelas, amostra, pathlib.Path("metricas/resultados"))
print("gravado:", arquivo)

gravado: metricas/resultados/comparacao_ouro.xlsx
